In [9]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
# 读取附件中的数据
attachment1 = pd.read_excel('附件1.xlsx') # 包含地块信息
attachment2 = pd.read_excel('附件2.xlsx') # 包含作物信息和 2023 年的统计数据
# 查看数据框列名，确保列名与代码一致
print(attachment1.columns)
print(attachment2.columns)
# 使用表格中的实际列名
# 地块信息：'地块名称', '地块类型', '地块面积/亩'
plots = attachment1[['地块名称', '地块类型', '地块面积/亩']].to_dict('records') # 作物信息：'作物名称', '作物类型', '种植面积/亩'
crops = attachment2[['作物名称', '作物类型', '种植面积/亩']].to_dict('records')

# 定义优化问题
model = LpProblem(name="Crop-Optimization", sense=LpMaximize)
# 决策变量：x[i][j][t] 表示第 t 年地块 i 种植作物 j 的面积
years = range(2024, 2031) # 从 2024 年到 2030 年
x = LpVariable.dicts("x", ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years), lowBound=0, cat="Continuous")
# 辅助变量：z[i][j][t] 表示实际销售的作物产量（不能超过总产量和预期销售量的最小值）
z = LpVariable.dicts("z", ((i['地块名称'], j['作物名称'], t) for i in  plots for j in crops for t in years), lowBound=0, cat="Continuous")
# 二进制变量：y[i][j][t] 表示第 t 年地块 i 是否种植作物 j（1 表示种植，0 表示不种植）
y = LpVariable.dicts("y", ((i['地块名称'], j['作物名称'], t) for i in plots for j in crops for t in years),  lowBound=0, upBound=1, cat="Binary")
# 目标函数：最大化收益
model += lpSum(z[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] - x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩'] for i in plots for j in crops for t in years)
# 约束条件
# 1. 每个地块每年的总种植面积不能超过其实际面积
for i in plots:
    for t in years:
        model += lpSum(x[i['地块名称'], j['作物名称'], t] for j in
                       crops) <= i['地块面积/亩']
# 2. 作物实际销售产量 z[i][j][t] 不能超过作物的总产量和预期销售量
for i in plots:
    for j in crops:
        for t in years:
            model += z[i['地块名称'], j['作物名称'], t] <= x[i['地块名称'], j['作物名称'], t] * j['种植面积/亩']
            model += z[i['地块名称'], j['作物名称'], t] <= j['种植面积/亩']

# 3. 不重茬约束：同一地块不能连续两年种植相同作物
# 使用二进制变量 y 来表示是否种植
for i in plots:
    for j in crops:
        for t in range(2025, 2031): # 确保 t 和 t-1 之间没有种植相同作物
            model += y[i['地块名称'], j['作物名称'], t] + y[i['地块名称'], j['作物名称'], t-1] <= 1
# 4. 每三年内必须种植一次豆类作物
for i in plots:
    for t in range(2024, 2028):
        model += lpSum(y[i['地块名称'], j['作物名称'], t+k] for j in
                       crops if j['作物类型'] == '豆类' for k in range(3)) >= 1
# 5. 面积约束：如果某地块某年种植某作物，则种植面积必须大于 0
for i in plots:
    for j in crops:
        for t in years:
            model += x[i['地块名称'], j['作物名称'], t] <= y[i['地块名称'], j['作物名称'], t] * i['地块面积/亩']
# 求解模型
status = model.solve()
# 输出结果
print(f"求解状态: {status}")
for i in plots:
    for j in crops:
        for t in years:
            if x[i['地块名称'], j['作物名称'], t].value() > 0:
                print(f"地块 {i['地块名称']} 在第 {t} 年种植 {j['作物名称']} {x[i['地块名称'], j['作物名称'], t].value()} 亩")

Index(['地块名称', '地块类型', '地块面积/亩', '说明 '], dtype='object')
Index(['种植地块', '作物编号', '作物名称', '作物类型', '种植面积/亩', '种植季次'], dtype='object')
求解状态: -1
地块 A1 在第 2024 年种植 小麦 1.0 亩
地块 A1 在第 2025 年种植 小麦 1.0 亩
地块 A1 在第 2026 年种植 小麦 1.0 亩
地块 A1 在第 2027 年种植 小麦 1.0 亩
地块 A1 在第 2028 年种植 小麦 1.0 亩
地块 A1 在第 2029 年种植 小麦 1.0 亩
地块 A1 在第 2030 年种植 小麦 1.0 亩
地块 A1 在第 2024 年种植 玉米 1.0 亩
地块 A1 在第 2025 年种植 玉米 1.0 亩
地块 A1 在第 2026 年种植 玉米 1.0 亩
地块 A1 在第 2027 年种植 玉米 1.0 亩
地块 A1 在第 2028 年种植 玉米 1.0 亩
地块 A1 在第 2029 年种植 玉米 1.0 亩
地块 A1 在第 2030 年种植 玉米 1.0 亩
地块 A1 在第 2024 年种植 玉米 1.0 亩
地块 A1 在第 2025 年种植 玉米 1.0 亩
地块 A1 在第 2026 年种植 玉米 1.0 亩
地块 A1 在第 2027 年种植 玉米 1.0 亩
地块 A1 在第 2028 年种植 玉米 1.0 亩
地块 A1 在第 2029 年种植 玉米 1.0 亩
地块 A1 在第 2030 年种植 玉米 1.0 亩
地块 A1 在第 2024 年种植 黄豆 1.0 亩
地块 A1 在第 2025 年种植 黄豆 1.0 亩
地块 A1 在第 2026 年种植 黄豆 1.0 亩
地块 A1 在第 2027 年种植 黄豆 1.0 亩
地块 A1 在第 2028 年种植 黄豆 1.0 亩
地块 A1 在第 2029 年种植 黄豆 1.0 亩
地块 A1 在第 2030 年种植 黄豆 1.0 亩
地块 A1 在第 2024 年种植 绿豆 1.0 亩
地块 A1 在第 2025 年种植 绿豆 1.0 亩
地块 A1 在第 2026 年种植 绿豆 1.0 亩
地块 A1 在第 2027 年种植 绿豆 1.

In [10]:
import pandas as pd
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus, value
# 读取地块数据和作物种植数据
land_data = pd.read_excel("附件1.xlsx") # 假设第一张表有地块编号、地块面积、地块类型
crop_data = pd.read_excel("附件2.xlsx") # 假设第二张表包含了 2023 年的种植情况
# 打印列名，确保我们使用正确的列名
print(land_data.columns)
# 假设这些列在 crop_data 中需要自己提供，手动定义或从另一个文件读取这些值
# acre_yield: 亩产量, expected_sales: 预期销售量, sale_price: 销售价格, cost: 种植成本
acre_yield = [500, 600, 550] # 假设每种作物的亩产量，示例数据
expected_sales = [10000, 15000, 12000] # 预期销售量，示例数据
sale_price = [2.5, 3.0, 4.0] # 每种作物的销售价格，示例数据
cost = [1.0, 1.2, 1.5] # 每种作物的种植成本，示例数据
# 创建线性规划模型
model = LpProblem("Crop_Optimization", LpMaximize)
# 假设有 3 个作物，使用列表长度确定作物数量
num_crops = len(acre_yield)

# 决策变量：x_ij_t 表示在第 t 年，地块 i 上种植作物 j 的面积
# i 为地块索引，j 为作物索引，t 为年份索引
x = LpVariable.dicts("x", [(i, j, t) for i in range(len(land_data))
                           for j in range(num_crops)
                           for t in range(2024, 2031)],
                     lowBound=0)
# 新的二进制变量 y_ij_t 表示某个地块 i 在第 t 年是否种植作物 j
y = LpVariable.dicts("y", [(i, j, t) for i in range(len(land_data))
                           for j in range(num_crops)
                           for t in range(2024, 2031)],
                     lowBound=0, upBound=1, cat="Binary")
# 辅助变量：用于表示 min 和 max
min_production = LpVariable.dicts("min_production", [(i, j, t) for i
                                                     in range(len(land_data))
                                                     for j
                                                     in range(num_crops)
                                                     for t
                                                     in range(2024, 2031)],
                                  lowBound=0)
excess_production = LpVariable.dicts("excess_production", [(i, j, t)
                                                           for i in range(len(land_data))

                                                           for j in range(num_crops)

                                                           for t in range(2024, 2031)],
                                     lowBound=0)
# 定义目标函数：包括正常销售收益和超额部分以 50%价格销售的收益，减去种植成本
profit = lpSum([min_production[(i, j, t)] * sale_price[j] +
                excess_production[(i, j, t)] * 0.5 * sale_price[j] -
                x[(i, j, t)] * cost[j]
                for i in range(len(land_data))
                for j in range(num_crops)
                for t in range(2024, 2031)])
model += profit
# 约束条件 1：每个地块的种植面积不能超过地块实际面积
for i in range(len(land_data)):
    for t in range(2024, 2031):
        model += lpSum([x[(i, j, t)] for j in range(num_crops)]) <= land_data['地块面积/亩'][i]




# 约束条件 2：不重茬种植（同一地块不能连续两年种植相同的作物）
for i in range(len(land_data)):
    for j in range(num_crops):
        for t in range(2025, 2031): # 从 2025 年开始检查不重茬
            model += y[(i, j, t)] + y[(i, j, t-1)] <= 1 # 不允许连续两年种植同一作物
# 约束条件 3：豆类作物三年内至少种植一次（假设豆类作物为作物编号 0 和 1）
for i in range(len(land_data)):
    for T in [2024, 2027]:
        model += lpSum([y[(i, j, t)] for j in range(2) # 假设作物 0 和作物 1 是豆类
    for t in range(T, T+3)]) >= 1 # 三年内至少种植一次豆类作物
# 约束条件 4：min_production <= 产量且不能超过预期销售量
for i in range(len(land_data)):
    for j in range(num_crops):
        for t in range(2024, 2031):
            # 总产量
            production = x[(i, j, t)] * acre_yield[j]
            # min_production 不能超过预期销售量，并且不能大于实际产量
            model += min_production[(i, j, t)] <= expected_sales[j]
            model += min_production[(i, j, t)] <= production
            # excess_production 是超出部分
            model += excess_production[(i, j, t)] == production - min_production[(i, j, t)]
# 约束条件 5：x 和 y 之间的关系，确保 x_ij_t 为 0 时，y_ij_t 也为 0
M = 100000 # 一个较大的常数
for i in range(len(land_data)):
    for j in range(num_crops):
        for t in range(2024, 2031):
            model += x[(i, j, t)] <= y[(i, j, t)] * M
# 求解模型
model.solve()
# 输出求解结果
print(f"Status: {LpStatus[model.status]}")
for v in model.variables():
    print(f"{v.name} = {v.varValue}")
# 输出最大化的目标函数值（总收益）
print(f"Total Profit: {value(model.objective)}")

Index(['地块名称', '地块类型', '地块面积/亩', '说明 '], dtype='object')
Status: Optimal
excess_production_(0,_0,_2024) = 0.0
excess_production_(0,_0,_2025) = 0.0
excess_production_(0,_0,_2026) = 0.0
excess_production_(0,_0,_2027) = 0.0
excess_production_(0,_0,_2028) = 0.0
excess_production_(0,_0,_2029) = 0.0
excess_production_(0,_0,_2030) = 0.0
excess_production_(0,_1,_2024) = 0.0
excess_production_(0,_1,_2025) = 21000.0
excess_production_(0,_1,_2026) = 0.0
excess_production_(0,_1,_2027) = 21000.0
excess_production_(0,_1,_2028) = 0.0
excess_production_(0,_1,_2029) = 21000.0
excess_production_(0,_1,_2030) = 0.0
excess_production_(0,_2,_2024) = 32000.0
excess_production_(0,_2,_2025) = 0.0
excess_production_(0,_2,_2026) = 32000.0
excess_production_(0,_2,_2027) = 0.0
excess_production_(0,_2,_2028) = 32000.0
excess_production_(0,_2,_2029) = 0.0
excess_production_(0,_2,_2030) = 32000.0
excess_production_(1,_0,_2024) = 0.0
excess_production_(1,_0,_2025) = 0.0
excess_production_(1,_0,_2026) = 0.0
excess_prod

In [11]:
import numpy as np
import pandas as pd
from pulp import *
# 1. 读取当前目录中的 Excel 文件
data1 = pd.read_excel('附件1.xlsx')
data2 = pd.read_excel('附件2.xlsx')
# 2. 清理列名，确保列名正确
data1.columns = data1.columns.str.strip() # 去除列名中的空格
data2.columns = data2.columns.str.strip()
# 3. 获取地块和作物信息
plots = data1['地块名称'].unique() # 使用 '地块名称' 来标识地块
crops = data2['作物名称'].unique() # 使用 '作物名称' 列来标识作物
# 4. 提取相关的参数
yield_per_mu = data2.set_index('作物名称')['种植面积/亩'].to_dict() #假设 '种植面积/亩' 表示产量
planting_cost = {} # 如果有种植成本信息，请在这里补充
expected_sales = {} # 如果有预期销售量信息，请在这里补充
price_per_unit = {} # 如果有销售价格信息，请在这里补充
# 获取地块的面积
land_area = data1.set_index('地块名称')['地块面积/亩'].to_dict() # 使用'地块面积/亩' 来获取地块面积
# 5. 定义蒙特卡洛模拟的不确定性
def simulate_uncertainty(base_value, percentage_range):
    return base_value * np.random.uniform(1 - percentage_range, 1 + percentage_range)
# 6. 进行线性规划的求解
def optimize_planting(yield_per_mu, planting_cost, expected_sales, price_per_unit):
    # 创建线性规划问题
    model = LpProblem("Crop_Planning", LpMaximize)
    # 决策变量，定义每年每块地每种作物的种植面积
    plant_area = LpVariable.dicts("Plant_Area",
                                  ((year, plot, crop) for year in
                                   range(2024, 2031) for plot in plots for crop in
                                   crops),
                                  lowBound=0, cat='Continuous')
    # 引入一个辅助变量，表示实际销售量，等于产量和预期销售量的最小值
    sales_amount = LpVariable.dicts("Sales_Amount",
                                    ((year, plot, crop) for year in
                                     range(2024, 2031) for plot in plots for crop in
                                     crops),
                                    lowBound=0, cat='Continuous')
    # 目标函数：最大化总收益
    model += lpSum([
        (sales_amount[(year, plot, crop)] *
         simulate_uncertainty(price_per_unit.get(crop, 0), 0.05) - plant_area[
             (year, plot, crop)] *
         simulate_uncertainty(planting_cost.get(crop, 0), 0.05))
        for year in range(2024, 2031) for plot in plots for crop in
        crops
    ])
    # 约束条件：
    # 1. 每块地的种植面积不能超过该地块的实际面积
    for year in range(2024, 2031):
        for plot in plots:
            model += lpSum([plant_area[(year, plot, crop)] for crop
                            in crops]) <= land_area[plot]
    # 2. 不重茬约束：同一地块连续两年不能种同一种作物
    for year in range(2025, 2031):
        for plot in plots:
            for crop in crops:
                model += plant_area[(year, plot, crop)] + plant_area[(year - 1, plot, crop)] <= 1
    # 3. 三年内必须种植一次豆类作物
    bean_crops = ['大豆', '绿豆'] # 假设这些是豆类作物
    for plot in plots:
        for t in range(2024, 2029, 3):
            # 添加存在性检查，避免访问不存在的变量
            model += lpSum([plant_area[(year, plot, crop)] for year in
                            range(t, t + 3) for crop in bean_crops if
                            crop in crops]) >= 1
    # 4. 实际销售量应该等于产量和预期销售量的最小值
    for year in range(2024, 2031):
        for plot in plots:
            for crop in crops:
                # 实际销售量不能超过产量
                model += sales_amount[(year, plot, crop)] <= plant_area[(year, plot, crop)] * simulate_uncertainty(  yield_per_mu.get(crop, 0), 0.1)
                # 实际销售量不能超过预期销售量
                model += sales_amount[(year, plot, crop)] <=  expected_sales.get(crop, 0)
    # 求解模型
    model.solve()
    # 检查求解状态，确保模型成功求解
    if LpStatus[model.status] != 'Optimal':
        print(f'Error: Model did not find an optimal solution. Status: {LpStatus[model.status]}')
        return None
    # 输出结果
    for v in model.variables():
        if v.varValue is not None and v.varValue > 0: # 确保varValue 不为 None
            print(f'{v.name} = {v.varValue}')
            # 返回总收益
    return value(model.objective)
# 7. 运行模拟
results = []
for simulation in range(100): # 运行 100 次模拟
    print(f'Running simulation {simulation + 1}')
    total_profit = optimize_planting(yield_per_mu, planting_cost, expected_sales, price_per_unit)
    if total_profit is not None:
        results.append(total_profit)


# 输出平均收益和方差（如果有成功的模拟）
if results:
    print(f'Average Profit: {np.mean(results)}')
    print(f'Profit Standard Deviation: {np.std(results)}')
else:
    print('No successful simulations found.')


Running simulation 1
Plant_Area_(2024,_'A1',_'绿豆') = 1.0
Plant_Area_(2024,_'A2',_'绿豆') = 1.0
Plant_Area_(2024,_'A3',_'绿豆') = 1.0
Plant_Area_(2024,_'B14',_'绿豆') = 1.0
Plant_Area_(2024,_'B3',_'绿豆') = 1.0
Plant_Area_(2024,_'B5',_'绿豆') = 1.0
Plant_Area_(2024,_'B6',_'绿豆') = 1.0
Plant_Area_(2024,_'B8',_'绿豆') = 1.0
Plant_Area_(2024,_'C6',_'绿豆') = 1.0
Plant_Area_(2024,_'D3',_'绿豆') = 1.0
Plant_Area_(2024,_'D6',_'绿豆') = 1.0
Plant_Area_(2024,_'D7',_'绿豆') = 1.0
Plant_Area_(2024,_'E11',_'绿豆') = 0.4
Plant_Area_(2024,_'E12',_'绿豆') = 0.6
Plant_Area_(2024,_'E14',_'绿豆') = 0.6
Plant_Area_(2024,_'E15',_'绿豆') = 0.6
Plant_Area_(2024,_'E16',_'绿豆') = 0.4
Plant_Area_(2024,_'E2',_'绿豆') = 0.4
Plant_Area_(2024,_'E6',_'绿豆') = 0.4
Plant_Area_(2024,_'E7',_'绿豆') = 0.4
Plant_Area_(2024,_'E8',_'绿豆') = 0.4
Plant_Area_(2024,_'E9',_'绿豆') = 0.6
Plant_Area_(2024,_'F1',_'绿豆') = 0.4
Plant_Area_(2024,_'F2',_'绿豆') = 0.4
Plant_Area_(2024,_'F4',_'绿豆') = 0.6
Plant_Area_(2025,_'A4',_'绿豆') = 1.0
Plant_Area_(2025,_'B10',_'绿豆') = 1.0


In [3]:
import pandas as pd
import numpy as np
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
# 1. 读取 Excel 文件中的数据
df_land = pd.read_excel('附件1.xlsx') # 附件 1，地块信息
df_crops = pd.read_excel('附件2.xlsx') # 附件 2，2023 年农作物种植数据
# 2. 清理列名，防止因列名中有空格等字符引起错误
df_land.columns = df_land.columns.str.strip() # 清除列名中的多余空格
df_crops.columns = df_crops.columns.str.strip()
# 3. 提取必要的数据
land_area = df_land['地块面积/亩'].values # 各地块的面积
crop_names = df_crops['作物名称'].values # 作物名称
# 以下使用假设数据，你需要根据实际数据进行调整
crop_prices = np.random.uniform(10, 20, len(crop_names)) # 作物的价格，假设为随机值
crop_costs = np.random.uniform(100, 200, len(crop_names)) # 作物种植成本
crop_yield = np.random.uniform(300, 500, len(crop_names)) # 作物的亩产量
crop_sales = np.random.uniform(1000, 5000, len(crop_names)) # 作物的预期销售量
n_land = len(land_area) # 地块数量
n_crops = len(crop_names) # 作物数量
years = range(2024, 2031) # 2024-2030 年
# 4. 创建优化问题
prob = LpProblem("Optimal_Crop_Plan", LpMaximize)
# 5. 定义决策变量
# x[i, j, t] 表示在第 t 年地块 i 上种植作物 j 的面积
x = LpVariable.dicts("x", ((i, j, t) for i in range(n_land) for j in
                           range(n_crops) for t in years), lowBound=0, cat='Continuous')
# 定义辅助变量 P[i, j, t] 表示实际销售的作物数量，必须小于或等于预期销售量和实际产量
P = LpVariable.dicts("P", ((i, j, t) for i in range(n_land) for j in
                           range(n_crops) for t in years), lowBound=0, cat='Continuous')
# 定义二元变量 z[i, j, t]，表示第 t 年在地块 i 上是否种植作物 j
z = LpVariable.dicts("z", ((i, j, t) for i in range(n_land) for j in range(n_crops) for t in years), 0, 1, cat='Binary')

# 6. 目标函数：最大化收益
objective = lpSum(
    P[i, j, t] * crop_prices[j] - x[i, j, t] * crop_costs[j]
    for i in range(n_land) for j in range(n_crops) for t in years
)
prob += objective # 将目标函数加入问题
# 7. 添加约束条件
# (1) 地块面积限制
for i in range(n_land):
    for t in years:
        prob += lpSum(x[i, j, t] for j in range(n_crops)) <= land_area[i] # 每块地的总种植面积不能超过该地块的面积
# (2) P[i, j, t] 受两方面限制：
# a. 不能超过预期销售量
# b. 不能超过实际产量（x[i, j, t] * crop_yield[j]）
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            prob += P[i, j, t] <= crop_sales[j] # 不超过预期销售量
    prob += P[i, j, t] <= x[i, j, t] * crop_yield[j] # 不超过实际产量
# (3) 不重茬约束：同一地块不能连续两年种植相同作物
# 通过二元变量 z[i, j, t] 实现不重茬
for i in range(n_land):
    for j in range(n_crops):
        for t in range(2025, 2031): # 从 2025 年开始，检查前一年的种植情况
            prob += z[i, j, t] + z[i, j, t-1] <= 1 # 如果 t 年种植作物 j，则 t-1 年不能种植同一作物
# (4) 确保 z[i, j, t] 与 x[i, j, t] 一致
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            # 如果种植作物 j 的面积大于 0，则 z[i, j, t] 为 1，否则为 0
            prob += x[i, j, t] <= z[i, j, t] * land_area[i]
            prob += x[i, j, t] >= z[i, j, t] * 0.01 # 小于 0.01 认为没种植

# (5) 豆类作物三年内必须种植一次
for i in range(n_land):
    for t in range(2024, 2031, 3): # 每三年检查一次
        prob += lpSum(z[i, j, t+k] for j in range(n_crops) if '豆类' in  crop_names[j] for k in range(3)) >= 1
# 8. 求解模型
prob.solve()
# 9. 输出结果
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            if x[i, j, t].varValue > 0:
                print(f"地块{i+1}, 作物: {crop_names[j]}, 年份: {t}, 面积: {x[i, j, t].varValue}亩")

地块1, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块2, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块3, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块4, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块5, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块6, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块7, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块8, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块9, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块10, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块11, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块12, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块13, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块14, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块15, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块16, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块17, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块18, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块19, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块20, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块21, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块22, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块23, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块24, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块25, 作物: 菠菜 , 年份: 2030, 面积: 7.3114692亩
地块26, 作物:

In [1]:
import numpy as np
import pandas as pd
from pulp import *
# 1. 读取当前目录中的 Excel 文件
data1 = pd.read_excel('附件1.xlsx')
data2 = pd.read_excel('附件2.xlsx')
# 2. 清理列名，确保列名正确
data1.columns = data1.columns.str.strip() # 去除列名中的空格
data2.columns = data2.columns.str.strip()
# 3. 获取地块和作物信息
plots = data1['地块名称'].unique() # 使用 '地块名称' 来标识地块
crops = data2['作物名称'].unique() # 使用 '作物名称' 列来标识作物
# 4. 提取相关的参数
yield_per_mu = data2.set_index('作物名称')['种植面积/亩'].to_dict() #假设 '种植面积/亩' 表示产量
planting_cost = {} # 如果有种植成本信息，请在这里补充
expected_sales = {} # 如果有预期销售量信息，请在这里补充
price_per_unit = {} # 如果有销售价格信息，请在这里补充
# 获取地块的面积
land_area = data1.set_index('地块名称')['地块面积/亩'].to_dict() # 使用'地块面积/亩' 来获取地块面积
# 5. 定义蒙特卡洛模拟的不确定性
def simulate_uncertainty(base_value, percentage_range):
    return base_value * np.random.uniform(1 - percentage_range, 1 +
                                          percentage_range)
# 6. 进行线性规划的求解
def optimize_planting(yield_per_mu, planting_cost, expected_sales,
                      price_per_unit):
    # 创建线性规划问题
    model = LpProblem("Crop_Planning", LpMaximize)
    # 决策变量，定义每年每块地每种作物的种植面积
    plant_area = LpVariable.dicts("Plant_Area",
                                  ((year, plot, crop) for year in
                                   range(2024, 2031) for plot in plots for crop in
                                   crops),
                                  lowBound=0, cat='Continuous')
    # 引入一个辅助变量，表示实际销售量，等于产量和预期销售量的最小值
    sales_amount = LpVariable.dicts("Sales_Amount",
                                    ((year, plot, crop) for year in
                                     range(2024, 2031) for plot in plots for crop in
                                     crops),
                                    lowBound=0, cat='Continuous')
    # 目标函数：最大化总收益
    model += lpSum([
        (sales_amount[(year, plot, crop)] *
         simulate_uncertainty(price_per_unit.get(crop, 0), 0.05) - plant_area[
             (year, plot, crop)] *
         simulate_uncertainty(planting_cost.get(crop, 0), 0.05))
        for year in range(2024, 2031) for plot in plots for crop in
        crops
    ])
    # 约束条件：
    # 1. 每块地的种植面积不能超过该地块的实际面积
    for year in range(2024, 2031):
        for plot in plots:
            model += lpSum([plant_area[(year, plot, crop)] for crop
                            in crops]) <= land_area[plot]
    # 2. 不重茬约束：同一地块连续两年不能种同一种作物
    for year in range(2025, 2031):
        for plot in plots:
            for crop in crops:
                model += plant_area[(year, plot, crop)] + plant_area[(year - 1, plot, crop)] <= 1
    # 3. 三年内必须种植一次豆类作物
    bean_crops = ['大豆', '绿豆'] # 假设这些是豆类作物
    for plot in plots:
        for t in range(2024, 2029, 3):
            # 添加存在性检查，避免访问不存在的变量
            model += lpSum([plant_area[(year, plot, crop)] for year in
                            range(t, t + 3) for crop in bean_crops if
                            crop in crops]) >= 1
    # 4. 实际销售量应该等于产量和预期销售量的最小值
    for year in range(2024, 2031):
        for plot in plots:
            for crop in crops:
                # 实际销售量不能超过产量
                model += sales_amount[(year, plot, crop)] <= plant_area[(year, plot, crop)] * simulate_uncertainty(
                    yield_per_mu.get(crop, 0), 0.1)
                # 实际销售量不能超过预期销售量
                model += sales_amount[(year, plot, crop)] <= expected_sales.get(crop, 0)
    # 求解模型
    model.solve()
    # 检查求解状态，确保模型成功求解
    if LpStatus[model.status] != 'Optimal': 
        print(f'Error: Model did not find an optimal solution. Status:  {LpStatus[model.status]}')
        return None
    # 输出结果
    for v in model.variables():
        if v.varValue is not None and v.varValue > 0: # 确保 varValue 不为 None
            print(f'{v.name} = {v.varValue}')
    # 返回总收益
    return value(model.objective)
# 7. 运行模拟
results = []
for simulation in range(100): # 运行 100 次模拟
    print(f'Running simulation {simulation + 1}')
    total_profit = optimize_planting(yield_per_mu, planting_cost,
                                     expected_sales, price_per_unit)
    if total_profit is not None:
        results.append(total_profit)

        # 输出平均收益和方差（如果有成功的模拟）
if results:
    print(f'Average Profit: {np.mean(results)}')
    print(f'Profit Standard Deviation: {np.std(results)}')
else:
    print('No successful simulations found.')

Running simulation 1
Plant_Area_(2024,_'A1',_'绿豆') = 1.0
Plant_Area_(2024,_'A2',_'绿豆') = 1.0
Plant_Area_(2024,_'A3',_'绿豆') = 1.0
Plant_Area_(2024,_'B14',_'绿豆') = 1.0
Plant_Area_(2024,_'B3',_'绿豆') = 1.0
Plant_Area_(2024,_'B5',_'绿豆') = 1.0
Plant_Area_(2024,_'B6',_'绿豆') = 1.0
Plant_Area_(2024,_'B8',_'绿豆') = 1.0
Plant_Area_(2024,_'C6',_'绿豆') = 1.0
Plant_Area_(2024,_'D3',_'绿豆') = 1.0
Plant_Area_(2024,_'D6',_'绿豆') = 1.0
Plant_Area_(2024,_'D7',_'绿豆') = 1.0
Plant_Area_(2024,_'E11',_'绿豆') = 0.4
Plant_Area_(2024,_'E12',_'绿豆') = 0.6
Plant_Area_(2024,_'E14',_'绿豆') = 0.6
Plant_Area_(2024,_'E15',_'绿豆') = 0.6
Plant_Area_(2024,_'E16',_'绿豆') = 0.4
Plant_Area_(2024,_'E2',_'绿豆') = 0.4
Plant_Area_(2024,_'E6',_'绿豆') = 0.4
Plant_Area_(2024,_'E7',_'绿豆') = 0.4
Plant_Area_(2024,_'E8',_'绿豆') = 0.4
Plant_Area_(2024,_'E9',_'绿豆') = 0.6
Plant_Area_(2024,_'F1',_'绿豆') = 0.4
Plant_Area_(2024,_'F2',_'绿豆') = 0.4
Plant_Area_(2024,_'F4',_'绿豆') = 0.6
Plant_Area_(2025,_'A4',_'绿豆') = 1.0
Plant_Area_(2025,_'B10',_'绿豆') = 1.0


In [ ]:
import pandas as pd
import numpy as np
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
# 1. 读取 Excel 文件中的数据
df_land = pd.read_excel('附件 1.xlsx') # 附件 1，地块信息
df_crops = pd.read_excel('附件 2.xlsx') # 附件 2，2023 年农作物种植数据
# 2. 清理列名，防止因列名中有空格等字符引起错误
df_land.columns = df_land.columns.str.strip() # 清除列名中的多余空格
df_crops.columns = df_crops.columns.str.strip()
# 3. 提取必要的数据
land_area = df_land['地块面积/亩'].values # 各地块的面积
crop_names = df_crops['作物名称'].values # 作物名称
# 以下使用假设数据，你需要根据实际数据进行调整
crop_prices = np.random.uniform(10, 20, len(crop_names)) # 作物的价格，假设为随机值
crop_costs = np.random.uniform(100, 200, len(crop_names)) # 作物种植成本
crop_yield = np.random.uniform(300, 500, len(crop_names)) # 作物的亩产量
crop_sales = np.random.uniform(1000, 5000, len(crop_names)) # 作物的预期销售量
n_land = len(land_area) # 地块数量
n_crops = len(crop_names) # 作物数量
years = range(2024, 2031) # 2024-2030 年
# 4. 创建优化问题
prob = LpProblem("Optimal_Crop_Plan", LpMaximize)
# 5. 定义决策变量
# x[i, j, t] 表示在第 t 年地块 i 上种植作物 j 的面积
x = LpVariable.dicts("x", ((i, j, t) for i in range(n_land) for j in
                           range(n_crops) for t in years), lowBound=0, cat='Continuous')
# 定义辅助变量 P[i, j, t] 表示实际销售的作物数量，必须小于或等于预期销售量和实际产量
P = LpVariable.dicts("P", ((i, j, t) for i in range(n_land) for j in
                           range(n_crops) for t in years), lowBound=0, cat='Continuous')
# 定义二元变量 z[i, j, t]，表示第 t 年在地块 i 上是否种植作物 j
z = LpVariable.dicts("z", ((i, j, t) for i in range(n_land) for j in
                           range(n_crops) for t in years), 0, 1, cat='Binary')
# 6. 目标函数：最大化收益
objective = lpSum(
    P[i, j, t] * crop_prices[j] - x[i, j, t] * crop_costs[j]
    for i in range(n_land) for j in range(n_crops) for t in years
)
prob += objective # 将目标函数加入问题
# 7. 添加约束条件
# (1) 地块面积限制
for i in range(n_land):
    for t in years:
        prob += lpSum(x[i, j, t] for j in range(n_crops)) <= land_area[i] # 每块地的总种植面积不能超过该地块的面积
# (2) P[i, j, t] 受两方面限制：
# a. 不能超过预期销售量
# b. 不能超过实际产量（x[i, j, t] * crop_yield[j]）
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            prob += P[i, j, t] <= crop_sales[j] # 不超过预期销售量
        prob += P[i, j, t] <= x[i, j, t] * crop_yield[j] # 不超过实际产量
# (3) 不重茬约束：同一地块不能连续两年种植相同作物
# 通过二元变量 z[i, j, t] 实现不重茬
for i in range(n_land):
    for j in range(n_crops):
        for t in range(2025, 2031): # 从 2025 年开始，检查前一年的种植情况
            prob += z[i, j, t] + z[i, j, t-1] <= 1 # 如果 t 年种植作物 j，则 t-1 年不能种植同一作物
# (4) 确保 z[i, j, t] 与 x[i, j, t] 一致
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            # 如果种植作物 j 的面积大于 0，则 z[i, j, t] 为 1，否则为 0
            prob += x[i, j, t] <= z[i, j, t] * land_area[i]
            prob += x[i, j, t] >= z[i, j, t] * 0.01 # 小于 0.01 认为没种植

# (5) 豆类作物三年内必须种植一次
for i in range(n_land):
    for t in range(2024, 2031, 3): # 每三年检查一次
        prob += lpSum(z[i, j, t+k] for j in range(n_crops) if '豆类' in
                      crop_names[j] for k in range(3)) >= 1
# 8. 求解模型
prob.solve()
# 9. 输出结果
for i in range(n_land):
    for j in range(n_crops):
        for t in years:
            if x[i, j, t].varValue > 0:
                print(f"地块{i+1}, 作物: {crop_names[j]}, 年份: {t}, 面积: {x[i, j, t].varValue}亩")